In [ ]:
from pathlib import Path
import json
import pickle

import numpy as np
import skimage
import sklearn
from tqdm.notebook import tqdm
from skimage.feature import multiscale_basic_features
from skimage.io import imread
from skimage.segmentation import relabel_sequential
from sklearn.ensemble import RandomForestClassifier
from scipy.ndimage import gaussian_gradient_magnitude
from skl2onnx import to_onnx

from segmentation_utils import normalize_by_quantiles


In [ ]:
in_path = '/home/david/Desktop/weihua_hp1_cluster_annotations/'

image_subdirectory = 'images'
mask_subdirectory = 'masks_edgesnap'

model_name = 'rf_hp1_clusters_003'

# calculate 2D features plane-by-plane, even for 3D data
do_plane_by_plane = False

# kwargs to be passed to multiscale_basic_features
feature_fun_kwargs = {'sigma_max': 4.0}

# how many pixels per class and image to sample
max_pixels_per_class = 100_000

# whether to sample pixels with replacement (potentially oversampling rare classes)
# if True, we will use max_pixels_per_class for each class
sample_with_replacement = True

# whether to relabel mask to have sequential 1, 2, ... labels
relabel_mask = True

# set to True if 0 means unannotated instead of bg
# classifier will be trained to predict idx - 1 (e.g. if 1 was bg & 2 cell, classifer will be trained with 0,1)
zero_is_unlabelled = False

# kwargs to be passed to RF constructor
rf_classifier_kwargs = {'n_estimators': 100, 'n_jobs': -1}

# list of image idxs to use for training (to use only a subset)
# e.g.: [0, 1, 3] will skip img2 (which could be used for verification)
# if None, will use all images
img_idxs_for_train = None

# quantiles for image normalization
# set to None to disable normalization
normalization_quantiles = (0.02, 0.995)

In [ ]:
image_path = Path(in_path) / image_subdirectory
mask_path = Path(in_path) / mask_subdirectory

# get image-mask pairs
image_files = sorted(image_path.glob('*.tif'))
mask_files = sorted(mask_path.glob('*.tif'))

# show for verification
list(zip(image_files, mask_files))

In [ ]:
## TODO: split feature generation, data point sampling (& mask loading -> relabel), training

images = []
masks = []

for image_file, mask_file in zip(image_files, mask_files):

    
    # load image and mask
    img = imread(image_file).astype(float) # TODO: normalize?

    if normalization_quantiles is not None:
        img = normalize_by_quantiles(img, normalization_quantiles)

    mask = imread(mask_file).astype(int)
    
    # relabel if necessary
    if relabel_mask:
        mask, _, _ = relabel_sequential(mask)

    images.append(img)
    masks.append(mask)

    print(f'loaded {image_file}.')

    # NOTE: removed, 
    # refine masks with snap-to-edge
    # if do_snap_to_edge:
    #     mask_ref = snap_labels_to_edge(mask, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)
    # else:
    #     mask_ref = mask

ndims = [img.ndim for img in images]

if not do_plane_by_plane and 2 in ndims:
    print('dataset includes 2D images, will process all images in 2D plane-by-plane.')
    do_plane_by_plane = True

In [ ]:
feature_maps = []

for img in tqdm(images):
   
    # calculate multiscale features (similar to ilastik, etc.)
    # ignore plane-by-plane if data is already 2D
    if (img.ndim != 2) and do_plane_by_plane:
        features = np.stack([multiscale_basic_features(img_i, **feature_fun_kwargs) for img_i in img])
    else:
        features = multiscale_basic_features(img, **feature_fun_kwargs)

    feature_maps.append(features)

In [ ]:
train_x = []
train_y = []

for idx in range(len(images)):

    features = feature_maps[idx]
    mask = masks[idx]

    # flatten mask and features
    features_flat = features.reshape((-1, features.shape[-1]))
    mask_flat = mask.ravel()

    for label in np.unique(mask_flat):

        # TODO: warn on no labels for given class?

        # skip unlabelled 0 class
        if zero_is_unlabelled and label == 0:
            continue

        # pixels of class
        selection = np.flatnonzero(mask_flat == label)

        # sampled selection of those pixels
        if sample_with_replacement:
            selection_to_keep = np.random.choice(selection, max_pixels_per_class, replace=True)
        else:
            selection_to_keep = np.random.choice(selection, min(max_pixels_per_class, len(selection)), replace=False)

        # add to training x, y
        train_x.append(features_flat[selection_to_keep])
        # NOTE: when we ignore unlabelled 0, we subtract 1 from target
        # so the final prediction will be 0, 1, ... if valid input classes were 1, 2, ...
        train_y.append(mask_flat[selection_to_keep] - (1 if zero_is_unlabelled else 0))

    print(f'prepared features & sampled data points for {image_files[idx]}.')

# concat into one dataset
train_x = np.concatenate(train_x)
train_y = np.concatenate(train_y)

# fit RF
model = RandomForestClassifier(**rf_classifier_kwargs)
model.fit(train_x, train_y)

In [ ]:
# files to save
model_filename = model_name + '_model.onnx'
model_filename_pkl = model_name + '_model.pkl'
info_filename = model_name + '_info.json'

# TODO: save 2D or 3D
# info about model and feature extraction
info_dict = {
    'feature_fun_kwargs': feature_fun_kwargs,
    'rf_kwargs': rf_classifier_kwargs,
    'is_2d': do_plane_by_plane,
    'model_file': model_filename,
    'model_file_pickle': model_filename_pkl,
    'sklearn_version': sklearn.__version__,
    'skimage_version': skimage.__version__,
}

if normalization_quantiles is not None:
    info_dict['normalization_quantiles'] = normalization_quantiles

# save info to JSON
with open(Path(in_path) / info_filename, 'w') as f:
    json.dump(info_dict, f)

# save model to ONNX
onx = to_onnx(model, train_x[:1])
with open(Path(in_path) / model_filename, "wb") as f:
    f.write(onx.SerializeToString())

# save model with pickle
with open(Path(in_path) / model_filename_pkl, "wb") as f:
    pickle.dump(model, f)

### Test segmentation

In [ ]:
idx = 4

img = images[idx]
mask = masks[idx]
features = feature_maps[idx]

features_flat = features.reshape((-1, features.shape[-1]))

In [ ]:
# predict with sklearn model in memory
mask_pred = model.predict(features_flat).reshape(img.shape)

**Alternative:** Predict with saved ONNX model

In [ ]:
import onnxruntime as rt

# files to load (same as save above)
model_filename = model_name + '_model.onnx'
model_filename_pkl = model_name + '_model.pkl'
info_filename = model_name + '_info.json'

sess = rt.InferenceSession(Path(in_path) / model_filename, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
pred_onx = sess.run([label_name], {input_name: features_flat.astype(np.float32)})[0]

mask_pred = pred_onx.reshape(img.shape)

In [ ]:
import napari

print(f'displaying segmentation for {image_files[idx]}.')

mask_pred_for_plot = mask_pred
# alternative: only select one class
# mask_pred_for_plot = (mask_pred == 2).astype(int)

# relabel to get different colors than GT mask in visualization
mask_pred_for_plot, _, _ = relabel_sequential(mask_pred_for_plot, np.max(mask) + 1)

# TODO: don't mix snap-to-edge in here
# refine_predicted = False
# if refine_predicted:
#     mask_pred_for_plot = snap_labels_to_edge(mask_pred_for_plot, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.view_image(img)
viewer.add_labels(mask)
viewer.add_labels(mask_pred_for_plot)

### Test snap-to-edge

Here, we use a single image + mask and show snap-to-edge results

In [ ]:
import napari
from snap_to_edge import snap_labels_to_edge

# TODO: don't do snap-to-edge here, preprocess in another recipe
do_snap_to_edge = False
snap_to_edge_radius = 1
snap_to_edge_ggm_sigma = 1.0

idx = 0

img = images[idx]
mask = masks[idx]

# refine mask with edge snap
mask_ref = snap_labels_to_edge(mask, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)
# relabel to start with higher indices (for visualization)
mask_ref, _, _ = relabel_sequential(mask_ref, np.max(mask) + 1)

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.view_image(img)
viewer.add_labels(mask)
viewer.add_labels(mask_ref)